<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/13_galeria_errores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Galería de errores: imagen, cajas y zoom

Comparar **la misma imagen** con su anotación y las cajas predichas por cada detector. La segunda fila amplía
**la misma zona** en todos los paneles: se ve dónde quedó desplazada la caja, qué anotación se perdió y qué
detección no tiene correspondencia. Debajo aparecen las decisiones de las CNN, que no producen cajas.

1. Abrir en Colab, **sin GPU**, y ejecutar todas las celdas.
2. Elegir hasta dos detectores, las CNN y un tipo de error. Por defecto se prioriza Jamaica.
3. Elegir la imagen y el error que se quiere ampliar. Activar «Mostrar aciertos» para ver también sus cajas.

Solo lee los resultados del equipo con `entorno.preparar(solo_lectura=True)`. No entrena, no ejecuta
inferencia, no crea archivos en Drive y no cambia datos ni split. Necesita las corridas completas y sus
`cajas_val.csv`; si faltan, informa qué no se pudo cargar, sin inventar detecciones.

In [ ]:
# Setup: correr siempre primero.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
else:
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pdi_tp import clasificacion, entorno, errores

rutas = entorno.preparar(solo_lectura=True)
tabla, anotaciones = clasificacion.preparar_tabla(rutas)


## Corridas disponibles

Las CNN deben tener `params.receta = "comun_v2"`. Los detectores se comparan a **640 px** por defecto.
Para revisar una corrida histórica de otro tamaño, activar la opción siguiente y volver a ejecutar desde
esta celda: esa inspección no permite atribuir diferencias solo a la arquitectura.

Un CSV de cajas vacío, con sus encabezados y puntajes cero, significa «sin detecciones». **Un archivo
faltante no significa eso:** la corrida queda fuera del visor con el motivo. También se comprueba que
las cajas y las probabilidades sean de la misma exportación y cubran el val oficial.

In [ ]:
INCLUIR_DETECTORES_ANTERIORES = False

inventario, predicciones, cajas = errores.cargar_revision(
    rutas, tabla, incluir_detectores_anteriores=INCLUIR_DETECTORES_ANTERIORES)
display(inventario[["corrida", "familia", "receta", "fecha", "disponible", "motivo"]])
if inventario.empty:
    print("No hay corridas guardadas en", rutas.resultados_equipo)
else:
    print("Disponibles:", int(inventario.disponible.sum()), "de", len(inventario))
print("Val se usó para elegir checkpoints y para reportar: esta revisión no es un test independiente.")


## Cómo leer las cajas

- **Azul, discontinua:** anotación de referencia, con identificador `R`.
- **Verde:** predicción que coincide con una anotación a la IoU elegida.
- **Rojo:** detección sin coincidencia, incluida una caja duplicada.
- **Naranja:** predicción cercana a una anotación, pero con IoU insuficiente. La línea une sus centros.
- **Violeta, discontinua:** anotación sin una predicción cercana.

`P` identifica una predicción y el número que la acompaña es su confianza. La **IoU** mide la superposición:
1 es coincidencia exacta y 0 es ninguna. El umbral inicial es **0.5** y la confianza inicial **0.25**.
Los pares naranjas son una pista de mala localización (IoU entre 0.1 y el umbral), no una confirmación
de que sean el mismo objeto. Siguen contando como FP y FN al umbral elegido.

El emparejamiento es uno a uno, por confianza descendente. Las cuentas TP/FP/FN sirven para inspeccionar
esa imagen; **no reemplazan el mAP** ni se guardan como resultados oficiales. El zoom usa los píxeles
originales de 256×256: ampliarlos no agrega detalle. Las cajas pequeñas se conservan y se excluye solo
Z01/línea 1 según los criterios v1. N06/N16 quedan ocultos por defecto por su referencia provisional.

In [ ]:
# Índice inicial y función de visualización, también utilizables sin controles interactivos.
disponibles = inventario.loc[inventario.disponible]
detectores = disponibles.loc[disponibles.familia == "detector", "corrida"].tolist()
clasificadores = disponibles.loc[disponibles.familia == "clasificador", "corrida"].tolist()

def elegir_por_modelo(corridas, modo_preferido):
    elegidas = {}
    for corrida in corridas:
        modelo = corrida.split("/")[0]
        if modelo not in elegidas or corrida.endswith("/" + modo_preferido):
            elegidas[modelo] = corrida
    return list(elegidas.values())

DETECTORES = elegir_por_modelo(detectores, "coco_completo")[:2]
CLASIFICADORES = elegir_por_modelo(clasificadores, "imagenet_finetuning")

def ver_imagen(archivo, diagnosticos, cnn, *, zona=0, mostrar_aciertos=False):
    fila = tabla.loc[tabla.archivo == archivo].iloc[0]
    if fila.pendiente_revision:
        print("REFERENCIA PROVISIONAL: revisar N06/N16; no concluir error del modelo sin revisar la anotación.")
    print("Fuente:", fila.fuente, "| Etiqueta original:", "con paneles" if fila.y_original else "sin paneles")
    zonas = errores.zonas_error(diagnosticos)
    zoom = zonas[zona][1] if 0 <= zona < len(zonas) else None
    if zoom is not None:
        print("Zoom:", zonas[zona][0])
    reales = anotaciones.loc[(anotaciones.archivo == archivo) & anotaciones.incluida, errores.COORDENADAS]
    figura = errores.mostrar_imagen(rutas, archivo, reales.to_numpy(float), diagnosticos,
        zoom=zoom, mostrar_aciertos=mostrar_aciertos,
        titulo="Revisión visual · referencia provisional" if fila.pendiente_revision else "Revisión visual de val")
    display(figura)
    plt.close(figura)
    seleccion = predicciones.loc[(predicciones.archivo == archivo) & predicciones.corrida.isin(cnn)].copy()
    if not seleccion.empty:
        seleccion["real"] = seleccion.y_real.map({0: "sin paneles", 1: "con paneles"})
        seleccion["decisión"] = seleccion.prediccion.map({0: "sin paneles", 1: "con paneles"})
        seleccion["resultado"] = np.where(seleccion.acierto.astype(bool), "Acierto",
            np.where(seleccion.y_real == 1, "FN: omitió la presencia", "FP: indicó paneles"))
        print("CNN: decisión por imagen al umbral guardado; no predicen cajas.")
        display(seleccion[["corrida", "real", "decisión", "probabilidad", "resultado"]].reset_index(drop=True))
    if not diagnosticos:
        print("No se seleccionaron detectores con cajas disponibles; se muestra la referencia y las CNN.")

indice, diagnosticos = errores.revisar_imagenes(tabla, anotaciones,
    {c: cajas[c] for c in DETECTORES}, predicciones.loc[predicciones.corrida.isin(CLASIFICADORES)])


## Visor

Los filtros de cajas se aplican a los detectores seleccionados; el filtro «Error CNN» usa las decisiones
guardadas de las CNN seleccionadas. Se muestran primero las imágenes con más errores entre esos modelos,
sin usar esta selección de ejemplos para estimar su rendimiento general.

Se pueden cambiar confianza e IoU para entender un caso. Eso no modifica el umbral guardado ni el
entrenamiento. La confianza no puede bajar del mínimo de exportación de las cajas seleccionadas.
Si una imagen no aparece, probar «Todo val», «Todas las imágenes» o incluir los casos pendientes.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if predicciones.empty:
    print("No hay predicciones disponibles. Revisar los motivos del inventario y las corridas en Drive.")
elif widgets is None:
    print("Controles no disponibles. Se muestra el primer error; la celda siguiente permite elegir un archivo.")
    casos = indice.loc[(indice.errores_cajas > 0) | (indice.errores_clasificacion > 0)]
    casos = casos.loc[~casos.pendiente_revision].sort_values(
        ["jamaica", "errores_cajas", "errores_clasificacion"], ascending=False)
    if not casos.empty:
        nombre = casos.archivo.iloc[0]
        ver_imagen(nombre, diagnosticos[nombre], CLASIFICADORES)
else:
    selector_det = widgets.SelectMultiple(options=detectores, value=tuple(DETECTORES),
        description="Detectores", rows=max(2, min(4, len(detectores))), layout=widgets.Layout(width="100%"))
    selector_cnn = widgets.SelectMultiple(options=clasificadores, value=tuple(CLASIFICADORES),
        description="CNN", rows=max(2, min(5, len(clasificadores))), layout=widgets.Layout(width="100%"))
    region = widgets.Dropdown(options=[("Jamaica", True), ("Todo val", False)], description="Región")
    categoria = widgets.Dropdown(options=[("Cualquier error", "error"),
        ("Cajas cercanas / mal ajustadas", "desajustadas"), ("Anotaciones omitidas", "omitidas"),
        ("Detecciones sin correspondencia", "falsas"), ("Error CNN por imagen", "cnn"),
        ("Todas las imágenes", "todas")], description="Ver", layout=widgets.Layout(width="70%"))
    confianza = widgets.FloatSlider(value=0.25, min=0.01, max=0.99, step=0.01,
        description="Confianza", continuous_update=False)
    iou = widgets.FloatSlider(value=0.5, min=0.15, max=0.95, step=0.05, description="IoU",
        continuous_update=False)
    pendientes = widgets.Checkbox(value=False, description="Incluir N06/N16 (referencia provisional)",
        indent=False, layout=widgets.Layout(width="100%"))
    aciertos = widgets.Checkbox(value=False, description="Mostrar cajas correctas", indent=False)
    archivo = widgets.Dropdown(options=[], description="Imagen", layout=widgets.Layout(width="100%"))
    zona = widgets.Dropdown(options=[], description="Zoom", layout=widgets.Layout(width="100%"))
    anterior = widgets.Button(description="Anterior")
    siguiente = widgets.Button(description="Siguiente")
    aviso = widgets.HTML()
    salida = widgets.Output()
    estado = dict(actualizando=False, indice=indice, diagnosticos=diagnosticos)

    def dibujar(cambio=None):
        if estado["actualizando"]:
            return
        with salida:
            salida.clear_output(wait=True)
            if archivo.value is None:
                print("No hay imágenes con estos filtros. Cambiar región, tipo de error o modelos.")
                return
            ver_imagen(archivo.value, estado["diagnosticos"][archivo.value], selector_cnn.value,
                       zona=zona.value if zona.value is not None else -1, mostrar_aciertos=aciertos.value)

    def cambiar_imagen(cambio=None):
        if estado["actualizando"]:
            return
        estado["actualizando"] = True
        zonas = errores.zonas_error(estado["diagnosticos"].get(archivo.value, {}))
        zona.options = [("Imagen completa", -1)] + [(nombre, i) for i, (nombre, _) in enumerate(zonas)]
        zona.value = 0 if zonas else -1
        estado["actualizando"] = False
        dibujar()

    def actualizar(cambio=None):
        if estado["actualizando"]:
            return
        if len(selector_det.value) > 2:
            aviso.value = "Elegir hasta dos detectores para mantener legibles la imagen y el zoom."
            archivo.disabled = zona.disabled = True
            with salida:
                salida.clear_output(wait=True)
                print("Elegir hasta dos detectores para actualizar la vista.")
            return
        minimos = disponibles.loc[disponibles.corrida.isin(selector_det.value), "confianza_minima"]
        if not minimos.empty and confianza.value < minimos.max():
            aviso.value = "La confianza elegida es menor que el mínimo con que se exportaron esas cajas."
            archivo.disabled = zona.disabled = True
            with salida:
                salida.clear_output(wait=True)
                print("Subir la confianza al mínimo de exportación para actualizar la vista.")
            return
        archivo.disabled = zona.disabled = False
        estado["actualizando"] = True
        try:
            casos, diag = errores.revisar_imagenes(tabla, anotaciones,
                {c: cajas[c] for c in selector_det.value},
                predicciones.loc[predicciones.corrida.isin(selector_cnn.value)],
                confianza=confianza.value, iou_min=iou.value)
            if region.value:
                casos = casos.loc[casos.jamaica]
            if not pendientes.value:
                casos = casos.loc[~casos.pendiente_revision]
            if categoria.value == "error":
                casos = casos.loc[(casos.errores_cajas > 0) | (casos.errores_clasificacion > 0)]
            elif categoria.value == "cnn":
                casos = casos.loc[casos.errores_clasificacion > 0]
            elif categoria.value != "todas":
                casos = casos.loc[casos[categoria.value] > 0]
            casos = casos.sort_values(["errores_cajas", "errores_clasificacion", "archivo"],
                                      ascending=[False, False, True])
            previo = archivo.value
            estado.update(indice=casos, diagnosticos=diag)
            archivo.options = casos.archivo.tolist()
            archivo.value = previo if previo in archivo.options else (archivo.options[0] if archivo.options else None)
            aviso.value = f"{len(casos)} imágenes con estos filtros. TP/FP/FN son cuentas por caja, no por imagen."
        finally:
            estado["actualizando"] = False
        cambiar_imagen()

    def avanzar(paso):
        if archivo.options:
            archivo.index = (archivo.index + paso) % len(archivo.options)

    for control in (selector_det, selector_cnn, region, categoria, confianza, iou, pendientes):
        control.observe(actualizar, names="value")
    archivo.observe(cambiar_imagen, names="value")
    zona.observe(dibujar, names="value")
    aciertos.observe(dibujar, names="value")
    anterior.on_click(lambda _: avanzar(-1))
    siguiente.on_click(lambda _: avanzar(1))
    display(widgets.VBox([selector_det, selector_cnn, widgets.HBox([region, categoria]),
        widgets.HBox([confianza, iou]), pendientes, aciertos, aviso, archivo, zona,
        widgets.HBox([anterior, siguiente]), salida]))
    actualizar()


## Abrir un archivo concreto

También se puede elegir por nombre sin usar los controles. Copiar un nombre de `indice.archivo` y volver
a ejecutar esta celda. Usa los detectores iniciales, confianza 0.25 e IoU 0.5; `ZONA = 0` amplía el primer
error y `ZONA = -1` muestra solo la imagen completa.

In [ ]:
ARCHIVO = None  # Por ejemplo, un nombre completo de indice.archivo.
ZONA = 0

if ARCHIVO is not None:
    if ARCHIVO not in diagnosticos:
        raise ValueError("Elegir un archivo del val oficial que aparezca en indice.archivo.")
    ver_imagen(ARCHIVO, diagnosticos[ARCHIVO], CLASIFICADORES, zona=ZONA, mostrar_aciertos=True)


## Qué registrar en el informe

Para cada ejemplo: archivo y fuente, modelo/modo, fecha y receta del inventario, umbral de confianza,
IoU, cajas R/P relevantes y lo que se observa (panel pequeño, sombra, techo parecido o caja desplazada).
Una apariencia visual no prueba la causa del error. Incluir también aciertos y desacuerdos entre modelos;
una galería seleccionada por errores no estima el rendimiento general.

Val eligió los checkpoints y también se usa para reportar. Jamaica tiene solo 23 imágenes; no concluir
superioridad a partir de unos pocos ejemplos. Para los números globales y sus intervalos, usar
`10_comparacion_final`. Esta galería conserva los resultados originales y permite inspeccionarlos.